In [ ]:
#!pip install -U transformers
#!pip install --upgrade gradio
#!pip install "huggingface_hub<1.0.0"
#!pip install huggingface_hub transformers

In [2]:
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import get_json_schema
import json
import re

In [3]:
model_name = "HuggingFaceTB/SmolLM3-3B"
device='cpu' 
# allowed entries Expected one of cpu, cuda (NVIDA), ipu, xpu, mkldnn, opengl(any graphcis card should use), opencl(parallel to cuda runs on anything that is amd (cuda) more generic), ideep, hip, ve, fpga, maia, xla, lazy, vulkan (like opengl - another graphic card language), mps, meta, hpu, mtia, privateuseone device type at start of device string: gpu

#With device='opengl' get error -- RuntimeError: PyTorch is not linked with support for opengl devices
#With device='opencl' get error -- RuntimeError: PyTorch is not linked with support for opencl devices
#With device='vulkan' get error -- RuntimeError: PyTorch is not linked with support for vulkan devices

#This is not a loading with Pytorch issue this is a pytorch can't run those issues according to ai overlord G. Mini at google need to use the ones specfiic to the chip set.

# Mac uses mps HAVE on list -- but keeps restarting the kernel so won't run entire process (my guess is just something something the handling doesn't work in jupiter notebooks). 
# AMD/Intel (Windows) uses Use ROCm (Linux) or DirectML (Windows/WSL).--- which isn't here ugh
# For NVIDIA GPUs: Use CUDA, which is the primary and most optimized backend for PyTorch. HAVE on list
# For Intel GPUs: Use the XPU backend via the Intel Extension for PyTorch (IPEX). HAVE on list

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

In [4]:
path="/Volumes/data/24.chdi.01-PHASE2/Kathryn_Connectomics/ZQ175_20260923-Female/Scalar_and_Volume/anovan_1/Genotype_AgeofTermination/Non_Erode/Subject_Data_Table.csv"
subject_data = pd.read_csv(path,delimiter='\t',low_memory=False,header=1) 
subject_data = subject_data.drop([0, 1]).reset_index(drop=True)

path="/Volumes/data/24.chdi.01-PHASE2/Kathryn_Connectomics/ZQ175_20260923-Female/Scalar_and_Volume/anovan_1/Genotype_AgeofTermination/Non_Erode/Bilateral_2/Group_Data_Table_Genotype.csv"
group_data = pd.read_csv(path,delimiter='\t',low_memory=False,header=1) 
group_data = group_data.drop([0, 1]).reset_index(drop=True)

path="/Volumes/data/24.chdi.01-PHASE2/Kathryn_Connectomics/ZQ175_20260923-Female/Scalar_and_Volume/anovan_1/Genotype_AgeofTermination/Non_Erode/Bilateral_2/Group_Statistical_Results_Genotype.csv"
group_stats = pd.read_csv(path,delimiter='\t',low_memory=False,header=0) 
group_stats = group_stats.drop([0, 1]).reset_index(drop=True)


available_documents = [
    {"title":"subject_data_table", "text": subject_data.to_json()},
    {"title":"group_data_table","text": group_data.to_json()},
    {"title":"group_statistical_results", "text":group_stats.to_json()}
]

In [5]:
# USER CREATED TOOLS 

def max(data:pd.DataFrame, column: str):
    """
    A function that takes a column and finds the maximum value in a given column

    Args:
        data: the data to work on
        column: The column to filter to
    """
    return a * b

def min (data:pd.DataFrame, column: str):
    """
    A function that takes a column and finds the minimum value in a given column

    Args:
        data: the data to work on
        column: The column to filter to
    """
    return a / b

available_tools = {
    "max": max,
    "min": min,
}

In [6]:
prompt=input("Input Your Prompt:")

Input Your Prompt: What region has the most change in FA?


In [7]:
messages_think = [
    {"role":"system","content": 'You are a research scientist working with a team of neuroscientists. They want to decipher key differences between scalar metrics (fractional anisotropy (fa), axial diffusivity (ad), radial diffusivity (rd), mean diffusivity (md), absolute volume (volume_mm3), and fractional volume (volume_frac – the volume of a given brain region as a percent of the total specimen’s brain volume)) of diffusion MRI studies of the mouse brain. The user will ask you questions about the data and your final response back to the user should be a scatter plot of the data that answers their question. The input data comes from 3 csv sheets: [1] subject_data_table [2] group_data_table [3] group_statistical_results. All data you utilize should be supported from data out of those three sheets.  Pull data directly from these sheets for plotting output results. If you have questions about what columns of the data to utilize ask the user, rather than guessing. If the user says something vague in their prompt like give me the top response, use the data columns associated with effect size measures (Cohen’s F, Cohen’s D, or Percent Change) to rank order the response prior to plotting.'},
    {"role":}
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages_think,
    tools=available_tools.values(),
    documents=available_documents,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
generated_ids = model.generate(**model_inputs, max_new_tokens=32768)

In [8]:
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
out=tokenizer.decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

pattern = r"(</?think>|</?tool_call>)"
split_out = re.split(pattern, out)

# Get Tools to Run out of the Message
begin_tool_call=0
result=[]
for line in split_out:
    if line == '<tool_call>':
        begin_tool_call=1
    elif line == '</tool_call>':
        begin_tool_call=0
    elif begin_tool_call==1 and (line != '<tool_call>' or line != '</tool_call>'):
        my_dict = json.loads(line)   
        tool_call_name = my_dict.get("name")
        tool_call_args= my_dict.get("arguments")
        if tool_call_name in available_tools:
            func_to_call = available_tools[tool_call_name]
            
            # Unpack the dictionary directly into the function parameters
            result = func_to_call(**tool_call_args)
            
        # Replace the tool call with the response!
        
if not result:
    result = out
            
print(result)

messages_think.append({"role":"assistant","content":result})

To determine which region has the most change in fractional anisotropy (FA), we need to calculate the percent change in FA for each region. This can be achieved by using the Percent Change function. 

To use the Percent Change function, we need the FA values for each subject and each region. We'll calculate the percent change for each region across all subjects and then find the region with the highest percent change.

Here are the function calls to calculate the percent change for each region:

1. Calculate the percent change for the Hippocampus:
   {"name": "percent_change", "arguments": {"data": {"subject_data_table": {"column": "hippocampus_fa"}, "group_data_table": {"column": "hippocampus_fa"}, "group_statistical_results": {"column": "percent_change_hippocampus"}}}}

2. Calculate the percent change for the Prefrontal Cortex:
   {"name": "percent_change", "arguments": {"data": {"subject_data_table": {"column": "prefrontal_cortex_fa"}, "group_data_table": {"column": "prefrontal_cort

In [10]:
type(group_stats)

pandas.core.frame.DataFrame

In [13]:
import pandas as pd
from datasets import Dataset
dataset = Dataset.from_pandas(group_stats)

ImportError: cannot import name 'HfFolder' from 'huggingface_hub' (/Users/kjh60/anaconda3/lib/python3.11/site-packages/huggingface_hub/__init__.py)